# EDA y modelo: ¿será un videojuego superventas?

<u>Estudios</u>: **Máster Universitario en Inteligencia Artificial** · <u>Asignatura</u>: **Lenguajes de Programación para IA** (UDIT)

**Dataset:** [Video Game Sales](https://www.kaggle.com/datasets/gregorut/videogamesales) de Kaggle (`vgsales.csv`), creado por Gregory Smith a partir de vgchartz.com: 16.598 videojuegos con más de 100.000 copias vendidas, con su plataforma, año, género, editora y ventas por región (en millones de unidades).

**Objetivo:**
1. Hacer un **análisis exploratorio (EDA)** del dataset.
2. Entrenar un **modelo sencillo** que prediga si un juego será **superventas** (1 millón de copias o más) usando solo información conocida **antes de su lanzamiento**: plataforma, género, editora y año.

**Ejecución:** ejecuta la celda 0.2, reinicia el kernel y usa `Run All`.

## 0. Preparación de entorno

### 0.1 Versión de Python

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Executable:", sys.executable)

### 0.2 Comprobar e instalar librerías

Si falta alguna librería en el kernel que estás usando, se instala en ese mismo entorno. Si instala algo, reinicia el kernel y usa `Run All`.

In [ ]:
import importlib.util
import subprocess

paquetes = {"pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn", "plotly": "plotly", "nbformat": "nbformat",
            "pymysql": "pymysql", "cryptography": "cryptography"}
faltan = [pip for modulo, pip in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltan])
    print("Instalación terminada. Reinicia el kernel (Restart) y ejecuta Run All.")
else:
    print("Todas las librerías necesarias están instaladas.")

### 0.3 Importamos los módulos necesarios

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

#### PARA EL RENDER DE LOS GRAFICOS ####
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # establece renderer

AZUL, NARANJA, GRIS = "#2a78d6", "#eb6834", "#8a8f98"
PLANTILLA = "plotly_white"

# 1. Análisis exploratorio (EDA)

## 1.1 Carga y primer vistazo

In [ ]:
def localizar_vgsales():
    """Busca vgsales.csv junto al notebook y en las carpetas habituales."""
    casa = Path.home()
    candidatos = [
        Path("vgsales.csv"),
        Path("..") / "vgsales.csv",
        casa / "OneDrive" / "Escritorio" / "LenguajeDeProgramacion_PabloManuelMartinGarcia_UDIT" / "vgsales.csv",
        casa / "Desktop" / "vgsales.csv",
        casa / "Downloads" / "vgsales.csv",
        casa / "Descargas" / "vgsales.csv",
    ]
    for ruta in candidatos:
        if ruta.is_file():
            return ruta
    raise FileNotFoundError("No se encontró vgsales.csv. Colócalo en la misma carpeta que el notebook.")


df = pd.read_csv(localizar_vgsales())
print("Dimensiones:", df.shape)
display(df.head(10))

In [ ]:
df.info()
display(df.describe().T.round(2))

## 1.2 Calidad de los datos: nulos y duplicados

Hay 271 juegos sin año y 58 sin editora (menos del 2 %). Se dejan como están: el modelo los imputará dentro de su pipeline. Además, el mismo juego aparece varias veces cuando salió en varias plataformas (cada fila es un juego **en una plataforma**).

In [ ]:
nulos = df.isna().sum()
display(nulos[nulos > 0].to_frame("nulos").assign(porcentaje=lambda t: (100 * t["nulos"] / len(df)).round(2)))

print("Filas duplicadas exactas:", df.duplicated().sum())
print("Juegos distintos (por nombre):", df["Name"].nunique(), "de", len(df), "filas")
print("Plataformas:", df["Platform"].nunique(), "· Géneros:", df["Genre"].nunique(), "· Editoras:", df["Publisher"].nunique())

## 1.3 Distribución de las ventas

Las ventas están **muy sesgadas**: la mayoría de juegos vende poco y unos pocos superventas venden muchísimo. Por eso el histograma usa escala logarítmica en el eje X. La línea marca el umbral de 1 millón que define un superventas.

In [ ]:
fig = px.histogram(
    df, x="Global_Sales", nbins=80, log_x=True,
    title="Distribución de las ventas globales por juego",
    labels={"Global_Sales": "Ventas globales (millones, escala log)"},
)
fig.update_traces(marker_color=AZUL)
fig.add_vline(x=1, line_dash="dash", line_color=NARANJA, annotation_text="1 millón (superventas)")
fig.update_layout(title_x=0.5, template=PLANTILLA, yaxis_title="Número de juegos", bargap=0.05)
fig.show()

print(f"Mediana: {df['Global_Sales'].median():.2f} M · Media: {df['Global_Sales'].mean():.2f} M · Máximo: {df['Global_Sales'].max():.2f} M")

## 1.4 Variable objetivo: superventas

Se define `superventas = 1` si el juego vende 1 millón o más. Solo el **12,5 %** de los juegos lo son: las clases están **desbalanceadas**. Esto obliga a usar métricas como el *recall*, el F1 o el AUC además de la *accuracy*: un modelo que dijera siempre "no" acertaría el 87,5 % sin servir para nada.

In [ ]:
df["superventas"] = (df["Global_Sales"] >= 1).astype(int)

conteo = df["superventas"].map({0: "No (< 1 M)", 1: "Sí (≥ 1 M)"}).value_counts().reset_index()
conteo.columns = ["superventas", "juegos"]
conteo["porcentaje"] = conteo["juegos"] / conteo["juegos"].sum()

fig = px.bar(conteo, x="superventas", y="juegos", color="superventas",
             color_discrete_map={"No (< 1 M)": GRIS, "Sí (≥ 1 M)": NARANJA},
             text=[f"{j:,} ({p:.1%})" for j, p in zip(conteo["juegos"], conteo["porcentaje"])],
             title="¿Cuántos juegos son superventas?")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(title_x=0.5, template=PLANTILLA, showlegend=False, xaxis_title=None, yaxis_title="Número de juegos")
fig.show()

## 1.5 Evolución temporal

El número de lanzamientos y las ventas crecen hasta 2008-2009 y luego caen. La caída final se debe sobre todo a cómo se construyó el dataset (ventas físicas recogidas en 2016; los juegos recientes llevan menos tiempo a la venta). Los años 2017 y 2020 tienen solo 4 juegos y se excluyen del gráfico.

In [ ]:
por_ano = (
    df.dropna(subset=["Year"]).query("Year <= 2016")
    .groupby("Year").agg(juegos=("Name", "size"), tasa_superventas=("superventas", "mean"))
    .reset_index()
)
fig = px.bar(por_ano, x="Year", y="juegos", title="Juegos lanzados por año",
             labels={"Year": "Año", "juegos": "Juegos lanzados"})
fig.update_traces(marker_color=AZUL)
fig.update_layout(title_x=0.5, template=PLANTILLA)
fig.show()

fig = px.line(por_ano, x="Year", y="tasa_superventas", markers=True,
              title="Porcentaje de superventas entre los juegos de cada año",
              labels={"Year": "Año", "tasa_superventas": "% de superventas"})
fig.update_traces(line_color=NARANJA)
fig.update_layout(title_x=0.5, template=PLANTILLA, yaxis_tickformat=".0%")
fig.show()

En la segunda mitad de los 80, dos de cada tres juegos registrados son superventas: de esa época el dataset recoge sobre todo los éxitos (sesgo de supervivencia). A partir de 2000 el porcentaje se estabiliza entre el 8 % y el 14 %. Por eso el **año** es una variable útil para el modelo.

## 1.6 Género y plataforma

Se compara **cuántos juegos** hay de cada género y **qué porcentaje** de ellos son superventas.

In [ ]:
por_genero = (
    df.groupby("Genre").agg(juegos=("Name", "size"), tasa_superventas=("superventas", "mean"))
    .sort_values("tasa_superventas").reset_index()
)
fig = px.bar(por_genero, x="tasa_superventas", y="Genre", orientation="h",
             text="tasa_superventas", custom_data=["juegos"],
             title="Porcentaje de superventas por género",
             labels={"tasa_superventas": "% de superventas", "Genre": "Género"})
fig.update_traces(marker_color=AZUL, texttemplate="%{text:.1%}", textposition="outside", cliponaxis=False,
                  hovertemplate="%{y}<br>Superventas: %{x:.1%}<br>Juegos: %{customdata[0]:,}<extra></extra>")
fig.update_layout(title_x=0.5, template=PLANTILLA, xaxis_tickformat=".0%", height=500)
fig.show()

In [ ]:
por_plataforma = (
    df.groupby("Platform").agg(juegos=("Name", "size"), tasa_superventas=("superventas", "mean"))
    .query("juegos >= 100").sort_values("tasa_superventas").reset_index()
)
fig = px.bar(por_plataforma, x="tasa_superventas", y="Platform", orientation="h",
             text="tasa_superventas", custom_data=["juegos"],
             title="Porcentaje de superventas por plataforma (plataformas con 100 juegos o más)",
             labels={"tasa_superventas": "% de superventas", "Platform": "Plataforma"})
fig.update_traces(marker_color=AZUL, texttemplate="%{text:.1%}", textposition="outside", cliponaxis=False,
                  hovertemplate="%{y}<br>Superventas: %{x:.1%}<br>Juegos: %{customdata[0]:,}<extra></extra>")
fig.update_layout(title_x=0.5, template=PLANTILLA, xaxis_tickformat=".0%", height=650)
fig.show()

Las sobremesas más recientes (PS4, Xbox One, X360, PS3) y algunas antiguas (SNES, Atari 2600) tienen las tasas más altas, en torno al 20 %. Las más bajas son Saturn y las portátiles de Sony (PS Vita y PSP), por debajo del 5 %, seguidas de DS y PC, que tienen catálogos enormes con muchos juegos pequeños. Entre géneros, Platform y Shooter son los que más éxitos tienen en proporción, y Adventure y Strategy los que menos.

## 1.7 Editoras

Hay 578 editoras, pero la mayoría publica muy pocos juegos. Se muestran las 15 con más juegos publicados y su porcentaje de superventas. Es la variable con más diferencias: unas editoras convierten en éxito una gran parte de lo que publican y otras casi nada.

In [ ]:
por_editora = (
    df.groupby("Publisher").agg(juegos=("Name", "size"), tasa_superventas=("superventas", "mean"))
    .nlargest(15, "juegos").sort_values("tasa_superventas").reset_index()
)
fig = px.bar(por_editora, x="tasa_superventas", y="Publisher", orientation="h",
             text="tasa_superventas", custom_data=["juegos"],
             title="Porcentaje de superventas de las 15 editoras con más juegos",
             labels={"tasa_superventas": "% de superventas", "Publisher": "Editora"})
fig.update_traces(marker_color=AZUL, texttemplate="%{text:.1%}", textposition="outside", cliponaxis=False,
                  hovertemplate="%{y}<br>Superventas: %{x:.1%}<br>Juegos: %{customdata[0]:,}<extra></extra>")
fig.update_layout(title_x=0.5, template=PLANTILLA, xaxis_tickformat=".0%", height=560)
fig.show()

print("Editoras con 30 juegos o más:", (df["Publisher"].value_counts() >= 30).sum())

## 1.8 Correlación entre ventas regionales

Las ventas de cada región están muy correlacionadas con las globales (`Global_Sales` es literalmente su suma). Por eso **no se usan como variables del modelo**: sería hacer trampa (*data leakage*), porque las ventas por región no se conocen antes de lanzar el juego y "contienen" la respuesta.

In [ ]:
columnas_ventas = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]
correlacion = df[columnas_ventas].corr().round(2)
fig = px.imshow(correlacion, text_auto=True, color_continuous_scale="Blues", zmin=0, zmax=1,
                title="Correlación entre las ventas por región")
fig.update_layout(title_x=0.5, height=500)
fig.show()

## 1.9 Conclusiones del EDA

- Las ventas están muy concentradas: la mediana es de solo 0,17 M por juego y solo el 12,5 % llega al millón. **La variable objetivo está desbalanceada.**
- El **año**, la **plataforma**, el **género** y sobre todo la **editora** muestran diferencias claras en el porcentaje de superventas, así que son buenas candidatas como variables.
- Las ventas regionales **no** se pueden usar como variables: son parte de la respuesta.
- `Publisher` tiene 578 valores distintos: para no crear cientos de columnas, las editoras con menos de 30 juegos se agruparán en una categoría "poco frecuente".

# 2. Modelo: predecir si un juego será superventas

Un modelo sencillo y rápido, sin búsqueda de hiperparámetros: una **regresión logística** dentro de un `Pipeline` con su preprocesamiento.

## 2.1 Variables y división train/test

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score, accuracy_score

num_features = ["Year"]
cat_features = ["Platform", "Genre", "Publisher"]
X = df[num_features + cat_features]
y = df["superventas"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "· Test:", X_test.shape)
print(f"Superventas en train: {y_train.mean():.1%} · en test: {y_test.mean():.1%}")

## 2.2 Pipeline y entrenamiento

- `Year`: imputación con la mediana y `StandardScaler`.
- `Platform`, `Genre`, `Publisher`: imputación con la moda y `OneHotEncoder`. Con `min_frequency=30`, las categorías con menos de 30 juegos se agrupan en una sola columna.
- `class_weight="balanced"`: da más peso a los superventas (la clase minoritaria) para que el modelo no se limite a decir siempre "no".

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30)),
    ]), cat_features),
])

modelo = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

auc_cv = cross_val_score(modelo, X_train, y_train, cv=5, scoring="roc_auc")
print(f"AUC en validación cruzada (5 folds): {auc_cv.mean():.3f} ± {auc_cv.std():.3f}")

modelo.fit(X_train, y_train)
print("Modelo entrenado. Variables tras el One-Hot Encoding:",
      len(modelo.named_steps["preprocessor"].get_feature_names_out()))

## 2.3 Evaluación en test

Se compara con un modelo de referencia (`DummyClassifier`) que siempre predice la clase mayoritaria ("no es superventas").

In [ ]:
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

referencia = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
y_ref = referencia.predict(X_test)

comparacion = pd.DataFrame({
    "accuracy": [accuracy_score(y_test, y_ref), accuracy_score(y_test, y_pred)],
    "F1 (superventas)": [f1_score(y_test, y_ref, zero_division=0), f1_score(y_test, y_pred)],
    "AUC": [0.5, roc_auc_score(y_test, y_prob)],
}, index=["Referencia (siempre 'no')", "Regresión logística"]).round(3)
display(comparacion)
print(classification_report(y_test, y_pred, target_names=["No superventas", "Superventas"]))

In [ ]:
etiquetas = ["No superventas", "Superventas"]
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(cm, x=etiquetas, y=etiquetas, text_auto=True, color_continuous_scale="Blues",
                labels={"x": "Predicho", "y": "Real", "color": "Juegos"}, title="Matriz de confusión (test)")
fig.update_layout(title_x=0.5, height=450)
fig.show()

fpr, tpr, _ = roc_curve(y_test, y_prob)
fig = px.area(x=fpr, y=tpr, title=f"Curva ROC (AUC = {roc_auc_score(y_test, y_prob):.3f})",
              labels={"x": "Tasa de falsos positivos", "y": "Tasa de verdaderos positivos (recall)"})
fig.update_traces(line_color=AZUL)
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line={"dash": "dash", "color": GRIS})
fig.update_layout(title_x=0.5, template=PLANTILLA, height=480, width=560)
fig.show()

**Cómo leer los resultados:**

- La referencia tiene más *accuracy* (87,5 %) pero no detecta **ni un solo** superventas. Por eso la *accuracy* no sirve aquí.
- El modelo detecta alrededor del **73 % de los superventas** (*recall*), a costa de muchas falsas alarmas: de cada 10 juegos que marca como superventas, unos 3 lo son de verdad (*precision* ≈ 0,28). Aun así, es más del doble que el 12,5 % que se obtendría eligiendo juegos al azar.
- El **AUC ≈ 0,81** indica que el modelo ordena bien los juegos: un superventas suele recibir más probabilidad que uno que no lo es (0,5 sería azar y 1 sería perfecto).

## 2.4 ¿Qué ha aprendido el modelo?

En una regresión logística, cada coeficiente indica cuánto sube (positivo) o baja (negativo) la probabilidad de ser superventas. Se muestran los 10 más positivos y los 10 más negativos.

In [ ]:
nombres = modelo.named_steps["preprocessor"].get_feature_names_out()
coeficientes = pd.Series(modelo.named_steps["classifier"].coef_[0], index=nombres)
coeficientes.index = (
    coeficientes.index.str.replace("cat__", "", regex=False)
    .str.replace("num__", "", regex=False)
    .str.replace("_infrequent_sklearn", " (poco frecuentes)", regex=False)
)
extremos = pd.concat([coeficientes.nsmallest(10), coeficientes.nlargest(10)]).sort_values().reset_index()
extremos.columns = ["variable", "coeficiente"]
extremos["efecto"] = np.where(extremos["coeficiente"] > 0, "Aumenta la probabilidad", "Reduce la probabilidad")

fig = px.bar(extremos, x="coeficiente", y="variable", orientation="h", color="efecto",
             color_discrete_map={"Aumenta la probabilidad": AZUL, "Reduce la probabilidad": NARANJA},
             title="Variables con más influencia en la predicción",
             labels={"coeficiente": "Coeficiente de la regresión logística", "variable": ""})
fig.update_layout(title_x=0.5, template=PLANTILLA, height=650, legend_title_text="")
fig.show()

## 2.5 Probar el modelo con juegos nuevos

Se le pide la probabilidad de ser superventas a tres juegos hipotéticos. Cambia los valores para probar otros.

In [ ]:
juegos_nuevos = pd.DataFrame({
    "Year": [2015, 2015, 2015],
    "Platform": ["PS4", "PS4", "PC"],
    "Genre": ["Shooter", "Shooter", "Strategy"],
    "Publisher": ["Activision", "Unknown", "Unknown"],
})
juegos_nuevos["probabilidad_superventas"] = modelo.predict_proba(juegos_nuevos)[:, 1].round(3)
display(juegos_nuevos)

## 2.6 Guardar el modelo en un archivo `.pkl`

Se guarda el pipeline completo (preprocesamiento + regresión logística) con `pickle` en `modelo_superventas.pkl`, junto al notebook. Después se vuelve a cargar y se comprueba que da las mismas predicciones. (Los apuntes usan `joblib`, que funciona igual; aquí se usa `pickle` porque el `.pkl` es lo que se guarda en MySQL.)

In [ ]:
import pickle

ruta_modelo = Path("modelo_superventas.pkl")
with open(ruta_modelo, "wb") as archivo:
    pickle.dump(modelo, archivo)
print(f"Modelo guardado en: {ruta_modelo.resolve()} ({ruta_modelo.stat().st_size / 1024:.0f} KB)")

with open(ruta_modelo, "rb") as archivo:
    modelo_cargado = pickle.load(archivo)

muestra = X_test.sample(5, random_state=42)
assert (modelo_cargado.predict(muestra) == modelo.predict(muestra)).all()
display(pd.DataFrame({
    "Name": df.loc[muestra.index, "Name"],
    "Platform": muestra["Platform"],
    "Genre": muestra["Genre"],
    "Publisher": muestra["Publisher"],
    "probabilidad_superventas": modelo_cargado.predict_proba(muestra)[:, 1].round(3),
    "prediccion": modelo_cargado.predict(muestra),
    "real": y_test.loc[muestra.index],
}))

## 2.7 Guardar el `.pkl` en MySQL (Docker)

El `.pkl` se guarda en una base de datos MySQL que corre en un contenedor Docker. Los archivos están en la carpeta `mysql_modelo/` (instrucciones en su `README.md`):

1. Arranca MySQL desde la carpeta `mysql_modelo`: `docker compose up -d`
2. Ejecuta esta celda. Usa las funciones de `mysql_modelo/db_modelos.py` para guardar el modelo como `LONGBLOB` en la tabla `modelos`, con un número de versión, sus métricas en test y la versión de scikit-learn.
3. Para usar el modelo desde fuera del notebook: `python usar_modelo_mysql.py --abrir` (en la carpeta `mysql_modelo`). El script lo descarga de MySQL, predice y genera las gráficas.

Si MySQL no está arrancado, la celda lo avisa sin detener el notebook.

In [ ]:
sys.path.append(str(Path("mysql_modelo").resolve()))

try:
    import db_modelos
    from sklearn.metrics import precision_score, recall_score

    metricas_test = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "auc": roc_auc_score(y_test, y_prob),
    }
    conexion = db_modelos.conectar()
    try:
        id_modelo, version = db_modelos.guardar_modelo(
            conexion, modelo, nombre="superventas_vgsales", algoritmo="LogisticRegression",
            variables=num_features + cat_features, metricas=metricas_test,
        )
        print(f"Modelo guardado en MySQL: id={id_modelo}, version={version}")
        display(db_modelos.listar_modelos(conexion))

        modelo_mysql, info = db_modelos.cargar_modelo(conexion, "superventas_vgsales")
        assert (modelo_mysql.predict(X_test) == y_pred).all()
        print(f"Comprobado: el modelo v{info['version']} descargado de MySQL predice igual que el original.")
    finally:
        conexion.close()
except ModuleNotFoundError as error:
    print(f"Falta una librería ({error.name}). Instálala con: pip install -r mysql_modelo/requirements.txt")
except Exception as error:
    print(f"No se pudo conectar con MySQL ({type(error).__name__}: {error}).")
    print("Arranca el contenedor con 'docker compose up -d' en la carpeta mysql_modelo y vuelve a ejecutar esta celda.")

# 3. Conclusiones

- **EDA:** las ventas están muy concentradas en pocos juegos, solo el 12,5 % supera el millón, y la editora, la plataforma, el género y el año muestran diferencias claras en la probabilidad de éxito.
- **Modelo:** con una regresión logística sencilla, sin ajustar hiperparámetros y entrenada en menos de un segundo, se obtiene un AUC de alrededor de 0,81. Detecta la mayoría de los superventas, aunque con bastantes falsas alarmas.
- **Limitaciones:** el dataset solo incluye juegos con más de 100.000 copias y tiene sesgo de supervivencia en los años antiguos. Además, un mismo juego aparece en varias plataformas, así que puede caer a la vez en train y en test.
- **Despliegue:** el pipeline entrenado se guarda como `modelo_superventas.pkl` y en MySQL (Docker). El script `mysql_modelo/usar_modelo_mysql.py` lo descarga de la base de datos, predice juegos nuevos y genera un informe con gráficas.
- **Mejoras posibles:** probar un `RandomForestClassifier`, ajustar el umbral de decisión según si importa más no perderse éxitos o evitar falsas alarmas, o añadir variables como la nota de la crítica.